# Chapter 4 &mdash; A DFA as a Goto-Based Program

**Concept 6 of the Chapter 4 decomposition:** *A DFA as a Goto-Based Program*

States are labels; transitions are gotos. DFA are assembly-level programs, and at that level goto is all there is.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter4/Concept-DFA-As-Goto-Program/Concept-DFA-As-Goto-Program.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
# Run me first.  Works on Colab and on a local Jove checkout.
import os, subprocess, sys

def _git(*a):
    r = subprocess.run(('git',) + a, capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else ''

REPO = 'https://github.com/ganeshutah/Jove'
try:                       # ---- Colab: clone once, pull thereafter ----
    import google.colab
    was = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD') if os.path.isdir('Jove') else ''
    if os.path.isdir('Jove') and not was:
        print('Jove: WARNING ./Jove exists but is not a git checkout -- left as is')
    elif was:
        _git('-C', 'Jove', 'pull', '-q', '--ff-only')
        now = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD')
        if now and now != was:
            print('Jove: PULLED  %s -> %s' % (was, now))
            print(_git('-C', 'Jove', 'log', '--oneline', was + '..' + now))
        else:
            print('Jove: PULLED  already current at %s' % (now or was))
    else:
        _git('clone', '-q', REPO, 'Jove')
        print('Jove: CLONED  at %s' % (_git('-C', 'Jove', 'rev-parse',
                                             '--short', 'HEAD') or '?'))
    JOVE = 'Jove'
except ImportError:        # ---- local: the checkout above Chapter<N>/ ----
    JOVE = next((p for p in ('../..', '../../..', '..', '.')
                 if os.path.isdir(os.path.join(p, 'jove'))), '../..')
    print('Jove: LOCAL   checkout at %s'
          % (_git('-C', JOVE, 'rev-parse', '--short', 'HEAD') or '?'))
sys.path.insert(0, JOVE)

# A session can already hold an OLDER jove in sys.modules.  The pull above
# updates the files on disk, but `import` would hand back the cached module --
# so a fixed library still behaves like the broken one.  Drop them first.
for _m in [k for k in list(sys.modules) if k == 'jove' or k.startswith('jove.')]:
    del sys.modules[_m]

from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *

import jove; print('Jove loaded from', list(jove.__path__)[0])

## 1. The idea


A DFA **is** a goto-based program: its states are **labels** and its transitions are
**goto statements** executed on specific inputs.

"Eeew, goto?!" &mdash; the advice against `goto` is advice about human-authored high-level
code. A DFA is a low-level program, much like assembly, where gotos are *the* control
mechanism because there is nothing else.

## 2. Definitions

### The machine

In [ ]:
D = md2mc('''DFA
I : 0 -> I
I : 1 -> F
F : 0 -> F
F : 1 -> I
''')

### The same thing written as labelled gotos

In [ ]:
def as_goto_program(D):
    print("start: goto %s" % D["q0"])
    for q in sorted(D["Q"]):
        print("%s:%s" % (q, "   # ACCEPTING" if q in D["F"] else ""))
        print("    if no input left: %s" % ("accept" if q in D["F"] else "reject"))
        for ch in sorted(D["Sigma"]):
            print("    read '%s' -> goto %s" % (ch, step_dfa(D, q, ch)))

## 3. Tests

The DFA, printed as a program.

In [ ]:
as_goto_program(D)

Running that program by hand matches `accepts_dfa`.

In [ ]:
def run_as_program(D, s):
    pc = D["q0"]
    for ch in s:
        pc = step_dfa(D, pc, ch)      # the goto
    return pc in D["F"]

tests = ['', '1', '10', '11', '101', '1001']
assert all(run_as_program(D, s) == accepts_dfa(D, s) for s in tests)
for s in tests:
    print("%-6r program says %-5s  accepts_dfa says %s"
          % (s, run_as_program(D, s), accepts_dfa(D, s)))

## 4. Exercises


1. The "program" has one variable, `pc`. What does that say about DFA memory?
2. Add a counter to `run_as_program`. Which machine class have you just built?
3. Rewrite the program with a `while` loop and a dictionary. Is it still a DFA?

In [ ]:
# Your work for the exercises above.

## 5. Where next

In [ ]:
# Previous / next, and a search box for all 245 concepts.
# Type a chapter (Chapter7, ch7) or words from a title (pumping, subset).
#
# Following a link opens a NEW Colab runtime. To pull another concept's
# definitions into THIS session instead:  load_here('Chapter7/Concept-...')
from jove.Nav import nav, load_here
nav(here='Chapter4/Concept-DFA-As-Goto-Program')